Getting the data

In [1]:
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_num_threads(2)

device = torch.device("cpu")
homework_dir = Path.cwd()
if "__file__" in globals():
    homework_dir = Path(__file__).resolve().parent
elif (homework_dir / "08-deep learning" / "08-homework").is_dir():
    homework_dir = homework_dir / "08-deep learning" / "08-homework"

data_dir = homework_dir / "data"
if not (data_dir / "train").is_dir() or not (data_dir / "test").is_dir():
    raise FileNotFoundError("Download and extract data.zip as shown in README.md.")

normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)
resize = transforms.Resize(
    (200, 200),
    interpolation=transforms.InterpolationMode.BILINEAR,
)
evaluation_transform = transforms.Compose([
    resize,
    transforms.ToTensor(),
    normalize,
])

train_dataset = datasets.ImageFolder(data_dir / "train", transform=evaluation_transform)
evaluation_dataset = datasets.ImageFolder(data_dir / "test", transform=evaluation_transform)

expected_classes = {"curly": 0, "straight": 1}
if (
    train_dataset.class_to_idx != expected_classes
    or evaluation_dataset.class_to_idx != expected_classes
    or Counter(train_dataset.targets) != {0: 410, 1: 390}
    or Counter(evaluation_dataset.targets) != {0: 103, 1: 98}
):
    raise ValueError("Unexpected dataset split. Check data.zip against the course checksum.")

print("Classes:", train_dataset.class_to_idx)
print("Training images:", len(train_dataset))
print("Evaluation images:", len(evaluation_dataset))

Classes: {'curly': 0, 'straight': 1}
Training images: 800
Evaluation images: 201


Q1. Loss function

In [2]:
criterion = nn.BCEWithLogitsLoss()
print(type(criterion).__name__)

# The model returns one logit per image, so the loss combines sigmoid and binary cross-entropy.
# Answer: nn.BCEWithLogitsLoss()

BCEWithLogitsLoss


Q2. Number of parameters

In [3]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = torch.relu(self.hidden(x))
        return self.output(x)


model = HairModel().to(device)
n_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", n_parameters)

# A 200x200 image becomes 198x198 after the convolution and 99x99 after pooling.
# Answer: 20073473

Trainable parameters: 20073473


Baseline training

In [4]:
batch_size = 20
epochs = 10

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(seed),
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)


def run_epoch(loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    loss_sum = 0.0
    correct = 0
    total = 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device).float().unsqueeze(1)

            if training:
                optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

            predictions = torch.sigmoid(logits.detach()) >= 0.5
            loss_sum += loss.item() * labels.size(0)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return loss_sum / total, correct / total


def train_model(loader, history_name):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "evaluation_loss": [],
        "evaluation_accuracy": [],
    }

    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = run_epoch(loader, optimizer)
        evaluation_loss, evaluation_accuracy = run_epoch(evaluation_loader)

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["evaluation_loss"].append(evaluation_loss)
        history["evaluation_accuracy"].append(evaluation_accuracy)

        print(
            f"Epoch {epoch:2d}/{epochs} | "
            f"train loss {train_loss:.4f}, accuracy {train_accuracy:.4f} | "
            f"evaluation loss {evaluation_loss:.4f}, accuracy {evaluation_accuracy:.4f}",
            flush=True,
        )

    path = homework_dir / history_name
    path.write_text(json.dumps(history, indent=2) + "\n", encoding="utf-8")
    return history


baseline = train_model(train_loader, "history_baseline.json")

Epoch  1/10 | train loss 0.6385, accuracy 0.6450 | evaluation loss 0.6010, accuracy 0.6567


Epoch  2/10 | train loss 0.5441, accuracy 0.7175 | evaluation loss 0.6003, accuracy 0.6517


Epoch  3/10 | train loss 0.4955, accuracy 0.7550 | evaluation loss 0.6405, accuracy 0.6219


Epoch  4/10 | train loss 0.4330, accuracy 0.7913 | evaluation loss 0.6274, accuracy 0.6915


Epoch  5/10 | train loss 0.4048, accuracy 0.8087 | evaluation loss 0.6293, accuracy 0.6617


Epoch  6/10 | train loss 0.3953, accuracy 0.8075 | evaluation loss 0.6332, accuracy 0.7114


Epoch  7/10 | train loss 0.3437, accuracy 0.8363 | evaluation loss 0.6895, accuracy 0.6716


Epoch  8/10 | train loss 0.2806, accuracy 0.8850 | evaluation loss 0.6497, accuracy 0.7214


Epoch  9/10 | train loss 0.2236, accuracy 0.9100 | evaluation loss 0.7114, accuracy 0.7164


Epoch 10/10 | train loss 0.1584, accuracy 0.9387 | evaluation loss 1.0212, accuracy 0.6368


Q3. Baseline training accuracy

In [5]:
median_accuracy = np.median(baseline["train_accuracy"])
print(f"Median training accuracy: {median_accuracy:.2f}")

# Answer: 0.81

Median training accuracy: 0.81


Q4. Baseline training loss

In [6]:
loss_std = np.std(baseline["train_loss"], ddof=0)
print(f"Population standard deviation of training loss: {loss_std:.3f}")

# Answer: 0.139

Population standard deviation of training loss: 0.139


Training with data augmentation

In [7]:
train_transform_augmented = transforms.Compose([
    resize,
    transforms.RandomRotation(50, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])

train_dataset_augmented = datasets.ImageFolder(
    data_dir / "train", transform=train_transform_augmented
)
train_loader_augmented = DataLoader(
    train_dataset_augmented,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(seed + 1),
)

# Continue with the model weights and SGD momentum from baseline training.
augmented = train_model(train_loader_augmented, "history_augmented.json")

Epoch  1/10 | train loss 0.6758, accuracy 0.6050 | evaluation loss 0.6552, accuracy 0.6716


Epoch  2/10 | train loss 0.6129, accuracy 0.6475 | evaluation loss 0.6608, accuracy 0.6269


Epoch  3/10 | train loss 0.5899, accuracy 0.6800 | evaluation loss 0.5787, accuracy 0.7114


Epoch  4/10 | train loss 0.5923, accuracy 0.6750 | evaluation loss 0.5679, accuracy 0.7114


Epoch  5/10 | train loss 0.5604, accuracy 0.7137 | evaluation loss 0.5680, accuracy 0.7214


Epoch  6/10 | train loss 0.5567, accuracy 0.7075 | evaluation loss 0.5510, accuracy 0.7015


Epoch  7/10 | train loss 0.5441, accuracy 0.7025 | evaluation loss 0.5288, accuracy 0.7413


Epoch  8/10 | train loss 0.5329, accuracy 0.7212 | evaluation loss 0.5401, accuracy 0.7114


Epoch  9/10 | train loss 0.5019, accuracy 0.7612 | evaluation loss 0.5631, accuracy 0.7363


Epoch 10/10 | train loss 0.5308, accuracy 0.7262 | evaluation loss 0.5405, accuracy 0.7264


Q5. Augmented evaluation loss

In [8]:
mean_loss = np.mean(augmented["evaluation_loss"])
print(f"Mean evaluation loss: {mean_loss:.3f}")

# Answer: 0.575

Mean evaluation loss: 0.575


Q6. Augmented evaluation accuracy

In [9]:
last_five_accuracy = np.mean(augmented["evaluation_accuracy"][-5:])
print(f"Mean evaluation accuracy for epochs 6-10: {last_five_accuracy:.2f}")

# Answer: 0.72

Mean evaluation accuracy for epochs 6-10: 0.72
